In [1]:
# Install required packages
!pip install -U groq python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: groq
    Found existing installation: groq 0.37.1
    Uninstalling groq-0.37.1:
      Successfully uninstalled groq-0.37.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.2 requires groq<1.0.0,>=0.30.0, but you have groq 1.4.0 which is incompatible.


In [2]:
import os, json, time
from dotenv import load_dotenv
from groq import Groq
load_dotenv()

groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)
print("OpenAI key loaded:   ", "✅" if groq_client    else "❌ Missing!")

OpenAI key loaded:    ✅


In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI
from pageindex.retrieve import get_document
from pageindex.page_index_md import submit_document

print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

✅ PageIndex client ready
✅ OpenAI client ready


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [ ]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks
from pageindex.page_index_md import get_tree
PDF_PATH = "mlresearchpaper.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: mlresearchpaper.pdf
✅ Uploaded!
📋 Document ID: pi-cmq2fks4300sg01qx65sb6b8f
   (Save this ID — you'll use it throughout the notebook)


In [ ]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [25]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Playing Atari with Deep Reinforcement Learning",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Playing Atari with Deep Reinforcement Learning\n\nVolodymyr Mnih Koray Kavukcuoglu David Silver Alex Graves Ioannis Antonoglou\nDaan Wierstra Martin Riedmiller\nDeepMind Technologies\n{vlad,koray,david,alex.graves,ioannis,daan,martin.riedmiller} @ deepmind.com\n",
  "text": "# Playing Atari with Deep Reinforcement Learning\n\nVolodymyr Mnih Koray Kavukcuoglu David Silver Alex Graves Ioannis Antonoglou\nDaan Wierstra Martin Riedmiller\nDeepMind Technologies\n{vlad,koray,david,alex.graves,ioannis,daan,martin.riedmiller} @ deepmind.com\n",
  "nodes": [
    {
      "title": "Abstract",
      "node_id": "0001",
      "page_index": 1,
      "summary": "## Abstract\n\nWe present the first deep learning model to successfully learn control policies directly from high-dimensional sensory input using reinforcement learning.

In [26]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Playing Atari with Deep Reinforcement Learning  (p.1)
  └─ [0001] Abstract  (p.1)
  └─ [0002] 1 Introduction  (p.1)
  └─ [0003] 2 Background  (p.2)
  └─ [0004] 3 Related Work  (p.3)
  └─ [0005] 4 Deep Reinforcement Learning  (p.4)
    └─ [0006] 4.1 Preprocessing and Model Architecture  (p.5)
  └─ [0007] 5 Experiments  (p.6)
    └─ [0008] 5.1 Training and Stability  (p.6)
    └─ [0009] 5.2 Visualizing the Value Function  (p.7)
    └─ [0010] 5.3 Main Evaluation  (p.7)
  └─ [0011] 6 Conclusion  (p.8)
  └─ [0012] References  (p.8)


In [27]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 13
   Each node = one retrievable section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [28]:
from groq import Groq
def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""
    
            You are given a query and a document's tree structure (like a Table of Contents).
            Your task: identify which node IDs most likely contain the answer to the query.
            Think step-by-step about which sections are relevant.

            Query: {query}

            Document Tree:
            {json.dumps(compressed_tree, indent=2)}

            Reply ONLY in this exact JSON format:
            {{
            "thinking": "<your step-by-step reasoning>",
            "node_list": ["node_id1", "node_id2"]
            }}
            
            """

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You must always return valid JSON only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except Exception:
        print(content)
        raise ValueError("Model returned invalid JSON")

    return json.loads(response.choices[0].message.content)
    

In [29]:
query = "What is the main purpose of this document also tell me which library and language they used in this research paper"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the main purpose of this document also tell me which library and language they used in this research paper

🧠 LLM Reasoning:
To answer the query, we need to identify the main purpose of the document and the library and language used in the research paper. The main purpose of the document is likely to be found in the abstract or introduction. The abstract (node_id: 0001) provides a brief overview of the paper, including the main contribution, which is the first deep learning model to successfully learn control policies directly from high-dimensional sensory input using reinforcement learning. The introduction (node_id: 0002) further elaborates on the background and motivation of the research. The library and language used in the research paper are not explicitly mentioned in the provided tree structure, but it can be inferred that the paper is about deep reinforcement learning, which is a subfield of machine learning, and the library and language used are likely to be r

---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [30]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [31]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
        Answer the question using ONLY the provided context.
        For every claim you make, cite the section title and page number in parentheses.
        Be concise and precise.

        Question: {query}

        Context:
        {context}

        Answer:"""
    
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    
    return response.choices[0].message.content

In [32]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [33]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is the main purpose of this document also tell me which library and programing language they used in this research paper",
    tree=pageindex_tree
)

🔍 Query: What is the main purpose of this document also tell me which library and programing language they used in this research paper

🧠 Reasoning: To answer the query, we need to identify the main purpose of the document and the library and programming language used in the research paper. The main purpose of the document is likely to be found in...
🎯 Retrieved node IDs: ['0001', '0002']
📄 Sections found: ['Abstract', '1 Introduction']

📝 Answer:
The main purpose of this document is to present a deep learning model that learns control policies directly from high-dimensional sensory input using reinforcement learning (Abstract | Page 1). The library used in this research is the Arcade Learning Environment (ALE) (1 Introduction | Page 1), and the programming language is not explicitly mentioned. However, the model is a convolutional neural network, trained with a variant of Q-learning, which is typically implemented in languages like Python or C++ (1 Introduction | Page 1).
